Qublitz Quantum Circuit Golf Optimizer

Circuit format:
    {"n_qubits": 2, "gates": [{"gate": "H", "qubits": [0]}, ...]}

Supported gates: H, X, Z, S, SDG, T, TDG, CX.
CX qubit order: [control, target].
Qubit 0 is the top/MSB wire.

Config format:
    {"w1": 1.0, "w2": 6.0, "wd": 0.2}

Includes a LOCAL checker, not the organizers' private tests.

In [ ]:
import copy
import math
import random

import numpy as np


SINGLE_QUBIT_GATES = frozenset(
    ("H", "X", "Z", "S", "SDG", "T", "TDG")
)

ALLOWED_GATES = SINGLE_QUBIT_GATES | {"CX"}

SELF_INVERSE = {"H", "X", "Z", "CX"}

INVERSE_PAIRS = {
    ("S", "SDG"),
    ("SDG", "S"),
    ("T", "TDG"),
    ("TDG", "T"),
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def can_cancel_redundancies(last_gate, next_gate):
    """Return True if two gates cancel."""

    # Self-inverse gates
    if last_gate == next_gate and last_gate in SELF_INVERSE:
        return True

    # Inverse gate pairs
    if (last_gate, next_gate) in INVERSE_PAIRS:
        return True

    return False


def replacement_gate(last_gate, next_gate):
    """Return the replacement for two combinable gates."""

    if last_gate == next_gate == "S":
        return "Z"

    if last_gate == next_gate == "T":
        return "S"

    if last_gate == next_gate == "SDG":
        return "Z"

    if last_gate == next_gate == "TDG":
        return "SDG"

    return None


def can_commute(gate_1, gate_2):
    """Return True when the two gates are known to commute."""

    # Case 1: Gates on disjoint qubits
    if set(gate_1["qubits"]).isdisjoint(gate_2["qubits"]):
        return True

    # Case 2: Two CX gates
    if gate_1["gate"] == "CX" and gate_2["gate"] == "CX":

        control_a, target_a = gate_1["qubits"]
        control_b, target_b = gate_2["qubits"]

        return (
            target_a != control_b
            and target_b != control_a
        )

    # Case 3: One CX and one single-qubit gate
    if gate_1["gate"] == "CX":
        cx_gate = gate_1
        other_gate = gate_2

    elif gate_2["gate"] == "CX":
        cx_gate = gate_2
        other_gate = gate_1

    else:
        # Conservatively assume overlapping 1q gates
        # do not commute.
        return False

    control, target = cx_gate["qubits"]
    other_qubit = other_gate["qubits"][0]

    # Diagonal gates commute with CX on the control
    if other_gate["gate"] in {"Z", "S", "SDG", "T", "TDG"}:
        if other_qubit == control:
            return True

    # X commutes with CX on the target
    if other_gate["gate"] == "X":
        if other_qubit == target:
            return True

    return False


# ===========================================
# Level 1: Self-Inverse Cancellation, and Paired-Substitutions
# ===========================================

def optimize_level_1(circuit):

    n_qubits = circuit["n_qubits"]
    gates = copy.deepcopy(circuit["gates"]) # defensive copy against original

    # One stack for each qubit
    qubit_stacks = [[] for _ in range(n_qubits)]

    removed = set() # allows us to reconstruct original formatting at end by just bypassing the ones that were removed and replacing the ones that are substitued  

    # gate_index = the index out of the gate_info lines they gave us
    # gate_info = each line with the gate letter and qubit array per gate
    for gate_index, gate_info in enumerate(gates):

        gate_type = gate_info["gate"]
        qubits = gate_info["qubits"]

        can_cancel = True # assumes allowed to cancel, queries failure throughout
        can_replace = True # assumes allowed to replace, queries failure throughout

        previous_index = None # index of the last gate (in gate_info) that was applied/ is in that qubit's stack
        replacement = None # the corresponding replacement gate reference for if S^2 or T^2 appear

        # Check every qubit affected by the current gate
        for qubit_stack in qubits:
            # qubit_stack is the index throughout the iteration that tracks which qubit's stack we are on
            current_stack = qubit_stacks[qubit_stack] # current_stack is the actual stack object belonging to the qubit of number qubit_stack

            # No previous gate on this qubit
            if len(current_stack) == 0: 
                can_cancel = False
                can_replace = False
                break # obviously if the stack is empty, there are no previous gates that can possible cancel or form identity pairs

            last_gate_index = current_stack[-1] # the index (from gate_info) of the last gate gate this qubit's stack had applied
            last_gate_info = gates[last_gate_index] # the corresponding gate_info line for that gate to get its symbol and qubit array

            # Gates must act on exactly the same qubits
            if last_gate_info["qubits"] != qubits: # if the qubit that the last gate was applied on != the qubit that this gate is applied on, they don't satisfy any of our cancellations/identities
                can_cancel = False
                can_replace = False
                break

            # Check cancellation
            cancel = can_cancel_redundancies(last_gate_info["gate"], gate_type) # checks if there are cancellable redundancies between the last gate and this one

            # Check replacement
            replacement = replacement_gate( last_gate_info["gate"], gate_type) # checks if there exists a replacement gate (substitution) betweenthe last gate and this one

            if not cancel and replacement is None: # if neither was possible, bypass the next step and just append this gate as normal
                can_cancel = False
                can_replace = False
                break

            if not cancel: # manually falsify the unsatisfied ones if only one was possible
                can_cancel = False

            if replacement is None: # manually falsify the unsatisfied ones if only one was possible
                can_replace = False

            # Both gates must refer to the same previous operation
            if previous_index is None:
                previous_index = last_gate_index # sets the previous gate to its gate_info line index if there wasn't a previous index set

            elif previous_index != last_gate_index: # if the previous index
                can_cancel = False
                can_replace = False
                break

        # Perform cancellation only AFTER checking all qubits
        if can_cancel:

            for qubit_stack in qubits:
                qubit_stacks[qubit_stack].pop()

            removed.add(previous_index) # marked so known not to print at end
            removed.add(gate_index) # marked so known not to print at end

        elif can_replace:

            gates[previous_index]["gate"] = replacement 

            removed.add(gate_index)

        else:

            for qubit_stack in qubits:
                qubit_stacks[qubit_stack].append(gate_index) # if nothing special, add gate as normally would

    # Reconstruct circuit after processing every gate
    optimized_gates = [copy.deepcopy(gate_info) for i, gate_info in enumerate(gates) if i not in removed]

    return {
        "n_qubits": n_qubits,
        "gates": optimized_gates
    }


# ============================================================
# Level 2: Commutative CX Cancelling
# ============================================================

def optimize_level_2(circuit):

    gates = copy.deepcopy(circuit["gates"])

    # Repeat until no more CX pairs can be canceled
    while True:

        CX_indices = []

        for i in range(len(gates)):
            if gates[i]["gate"] == "CX":
                CX_indices.append(i)

        removed = set()

        # Compare every possible pair of CX gates
        for i in range(len(CX_indices)):

            for j in range(i + 1, len(CX_indices)):

                first_CX = CX_indices[i]
                second_CX = CX_indices[j]

                # Don't reuse canceled gates
                if first_CX in removed or second_CX in removed:
                    continue

                # Control and target must match
                if (
                    gates[first_CX]["qubits"]
                    != gates[second_CX]["qubits"]
                ):
                    continue

                # Check every intervening gate
                all_commute = True

                for k in range(first_CX + 1, second_CX): # checks if all of the gates between CX's are commutative

                    if k in removed: # skip removed CX indices because already dealt with
                        continue

                    if not can_commute(gates[first_CX], gates[k]):
                        all_commute = False
                        break

                if all_commute:

                    removed.add(first_CX)
                    removed.add(second_CX)

                    break

        # No further cancellations
        if len(removed) == 0:
            break

        # Remove canceled CX gates
        gates = [
            gate
            for index, gate in enumerate(gates)
            if index not in removed
        ]

    return {
        "n_qubits": circuit["n_qubits"],
        "gates": gates
    }


# =========================
# Level 3: Triple-gate identities
# =========================

def optimize_level_3(circuit):

    gates = copy.deepcopy(circuit["gates"])
    n_qubits = circuit["n_qubits"]

    qubit_stacks = [[] for _ in range(n_qubits)]

    removed = set()

    TRIPLET_REPLACEMENTS = {
        ("H", "X", "H"): "Z",
        ("H", "Z", "H"): "X",
    } # see pattern lower down; TRIPLET_REPLACEMENTS is a tuple-key:value dictionary storing hashable triplet patterns as tuples

    for gate_index, gate_info in enumerate(gates):

        gate_type = gate_info["gate"]
        qubits = gate_info["qubits"]

        # Only check triplets of single-qubit gates
        if len(qubits) == 1:

            q = qubits[0] # the qubit the gate's being applied to
            stack = qubit_stacks[q] # the stack of said qubit

            # Need two previous gates plus current gate
            if len(stack) >= 2:

                first_index = stack[-2]
                second_index = stack[-1]

                pattern = (gates[first_index]["gate"], gates[second_index]["gate"], gate_type) 

                replacement = TRIPLET_REPLACEMENTS.get(pattern) # looks within TRIPLET_REPLACEMENTS  for the pattern we formed above of the last 2 and current gates

                if replacement is not None: # it returns none if the pattern doenst match any of the TRIPLET_REPLACEMENTS

                    # Replace first H with X or Z
                    gates[first_index]["gate"] = replacement

                    # Remove middle gate and final H
                    removed.add(second_index)
                    removed.add(gate_index)

                    # Remove middle gate from stack
                    # Keep first_index because it now represents
                    # the replacement gate
                    stack.pop()

                    continue

        # Otherwise, record current gate on affected qubits
        for q in qubits:
            qubit_stacks[q].append(gate_index)

    optimized_gates = [
        gate
        for i, gate in enumerate(gates)
        if i not in removed
    ]

    return {
        "n_qubits": n_qubits,
        "gates": optimized_gates
    }


# ============================================================
# Main Optimizer: Cost-aware multi-strategy optimization
# ============================================================

def optimize(circuit, config):
    """
    Return an equivalent optimized circuit.

    Does not mutate the original circuit or config.

    Tries two optimization orders and keeps the cheapest
    equivalent result.
    """

    validate_circuit(circuit)
    validate_config(config)

    best = copy.deepcopy(circuit) # points to original array for now til most optimized option is found

    while True:

        previous_best = copy.deepcopy(best)

        # Strategy A: Prioritizing pairwise cancellation -- we run 1 (priority) then 3 to cancel all identities, then 2 to commute out any CXs, the 1 again (priority) to cancel remaining hecklers
        candidate_a = copy.deepcopy(best)

        candidate_a = optimize_level_1(candidate_a)
        candidate_a = optimize_level_3(candidate_a)
        candidate_a = optimize_level_2(candidate_a)
        candidate_a = optimize_level_1(candidate_a)


        # Strategy B: Prioritizing triplet identities -- we run 3 (priority) then 1 to cancel all identities, then 2 to commute out any CXs, then 3 again (priority), then 1 again to cancel remaining hecklers
        candidate_b = copy.deepcopy(best)

        candidate_b = optimize_level_3(candidate_b)
        candidate_b = optimize_level_1(candidate_b)
        candidate_b = optimize_level_2(candidate_b)
        candidate_b = optimize_level_3(candidate_b)
        candidate_b = optimize_level_1(candidate_b)

        # Compare only equivalent candidates
        candidates = [best] # initializes to the original, because if both candidates end up being worse than the original they wont be accepted if we initialize the candidates' best to the original

        for candidate in (candidate_a, candidate_b):

            if equivalent_up_to_global_phase(best, candidate): # checks to ensure both candidates produced equivalent gate sequences (Ucandidate = e^iphi Ubest)
                candidates.append(candidate) # if its chill and equivalent then it gets appended

        # Choose the cheapest circuit
        best = min(
            candidates,
            key=lambda c: circuit_cost(c, config)
        )
        # min takes smallest arg, but all of our args are dictionaries. Thus, we use a comparison criterion--key=lambda. 
        # 
        #  lambda c: circuit_cost(c, config)
        #  is equivalent to writing :
        # def calculate_candidate_cost(c):
        #   return circuit_cost(c, config)
        #
        # its a shorthand for writing a function. For both cases (candidate a and b) it receives a circuit c, calculates its cost using config, and then returns that cost.
        # the min takes the minimum cost of the the two

        # Stop when a complete round makes no improvement
        if best == previous_best:
            break

    return best


# ============================================================
# Circuit Validation (given to us)
# ============================================================

def validate_circuit(circuit):
    """Raise ValueError if a circuit violates the public format."""

    if not isinstance(circuit, dict) or set(circuit) != {
        "n_qubits", "gates"
    }:
        raise ValueError(
            "Circuit must have exactly 'n_qubits' and 'gates'."
        )

    n = circuit["n_qubits"]

    if type(n) is not int or not 1 <= n <= 6:
        raise ValueError(
            "n_qubits must be an integer from 1 through 6."
        )

    gates = circuit["gates"]

    if not isinstance(gates, list):
        raise ValueError("gates must be a list.")

    for i, operation in enumerate(gates):

        if not isinstance(operation, dict) or set(operation) != {
            "gate", "qubits"
        }:
            raise ValueError(
                f"Gate {i}: expected exactly 'gate' and 'qubits'."
            )

        name = operation["gate"]
        qubits = operation["qubits"]

        if not isinstance(name, str) or name not in ALLOWED_GATES:
            raise ValueError(
                f"Gate {i}: unsupported gate {name!r}."
            )

        if not isinstance(qubits, list):
            raise ValueError(
                f"Gate {i}: qubits must be a list."
            )

        arity = 2 if name == "CX" else 1

        if len(qubits) != arity:
            raise ValueError(
                f"Gate {i}: {name} requires {arity} qubit(s)."
            )

        if any(
            type(q) is not int or not 0 <= q < n
            for q in qubits
        ):
            raise ValueError(
                f"Gate {i}: invalid qubit index."
            )

        if len(set(qubits)) != len(qubits):
            raise ValueError(
                f"Gate {i}: qubit indices must be distinct."
            )

    return True


def validate_config(config):
    """Check nonnegative finite resource weights."""

    if not isinstance(config, dict) or set(config) != {
        "w1", "w2", "wd"
    }:
        raise ValueError(
            "config must have exactly 'w1', 'w2', and 'wd'."
        )

    for key in ("w1", "w2", "wd"):

        value = config[key]

        if isinstance(value, bool) or not isinstance(
            value, (int, float)
        ):
            raise ValueError(
                f"{key} must be a real number."
            )

        if not math.isfinite(value) or value < 0:
            raise ValueError(
                f"{key} must be finite and nonnegative."
            )

    if not any(
        config[key] > 0
        for key in ("w1", "w2", "wd")
    ):
        raise ValueError(
            "At least one weight must be positive."
        )

    return True


# ============================================================
# Quantum Gate Matrices (given to us)
# ============================================================

_ROOT2 = math.sqrt(2)

_ONE_QUBIT_MATRICES = {
    "H": np.array(
        [[1, 1], [1, -1]],
        dtype=complex
    ) / _ROOT2,

    "X": np.array(
        [[0, 1], [1, 0]],
        dtype=complex
    ),

    "Z": np.array(
        [[1, 0], [0, -1]],
        dtype=complex
    ),

    "S": np.diag(
        [1, 1j]
    ).astype(complex),

    "SDG": np.diag(
        [1, -1j]
    ).astype(complex),

    "T": np.diag(
        [1, np.exp(1j * np.pi / 4)]
    ).astype(complex),

    "TDG": np.diag(
        [1, np.exp(-1j * np.pi / 4)]
    ).astype(complex),
}


def _full_gate_matrix(n, operation):
    """Matrix for one gate, with qubit 0 as MSB."""

    name = operation["gate"]
    qubits = operation["qubits"]

    if name != "CX":

        result = np.array([[1]], dtype=complex)

        for q in range(n):

            factor = (
                _ONE_QUBIT_MATRICES[name]
                if q == qubits[0]
                else np.eye(2, dtype=complex)
            )

            result = np.kron(result, factor)

        return result

    control, target = qubits

    dimension = 1 << n

    result = np.zeros(
        (dimension, dimension),
        dtype=complex
    )

    control_bit = 1 << (n - 1 - control)
    target_bit = 1 << (n - 1 - target)

    for input_state in range(dimension):

        output_state = (
            input_state ^ target_bit
            if input_state & control_bit
            else input_state
        )

        result[output_state, input_state] = 1

    return result


def circuit_unitary(circuit):
    """Calculate the complete circuit unitary."""

    validate_circuit(circuit)

    n = circuit["n_qubits"]

    result = np.eye(
        1 << n,
        dtype=complex
    )

    for operation in circuit["gates"]:
        result = _full_gate_matrix(n, operation) @ result

    return result


def equivalent_up_to_global_phase(first, second, atol=1e-9):
    """Check equivalence up to global phase."""

    validate_circuit(first)
    validate_circuit(second)

    if first["n_qubits"] != second["n_qubits"]:
        return False

    a = circuit_unitary(first)
    b = circuit_unitary(second)

    pivot = np.unravel_index(
        np.argmax(np.abs(a)),
        a.shape
    )

    phase = b[pivot] / a[pivot]

    return (
        np.isclose(
            abs(phase),
            1.0,
            atol=atol,
            rtol=0.0
        )
        and np.allclose(
            b,
            phase * a,
            atol=atol,
            rtol=0.0
        )
    )


# ============================================================
# Circuit metrics/ cost (given to us)
# ============================================================

def circuit_metrics(circuit):
    """Count gates and minimum circuit depth."""

    validate_circuit(circuit)

    available = [0] * circuit["n_qubits"]

    one_qubit = 0
    two_qubit = 0

    for operation in circuit["gates"]:

        qubits = operation["qubits"]

        end = 1 + max(
            available[q]
            for q in qubits
        )

        for q in qubits:
            available[q] = end

        if operation["gate"] == "CX":
            two_qubit += 1
        else:
            one_qubit += 1

    return {
        "n_1q": one_qubit,
        "n_2q": two_qubit,
        "depth": max(available)
    }


def circuit_cost(circuit, config):
    """Weighted circuit cost."""

    validate_config(config)

    metrics = circuit_metrics(circuit)

    return (
        config["w1"] * metrics["n_1q"]
        + config["w2"] * metrics["n_2q"]
        + config["wd"] * metrics["depth"]
    )


def evaluate_optimization(original, optimized, config):
    """Return correctness and resource comparisons."""

    equivalent = equivalent_up_to_global_phase(
        original,
        optimized
    )

    before = circuit_cost(original, config)
    after = circuit_cost(optimized, config)

    return {
        "equivalent": equivalent,
        "input_metrics": circuit_metrics(original),
        "output_metrics": circuit_metrics(optimized),
        "input_cost": before,
        "output_cost": after,
        "cost_reduction": before - after,
    }


# ============================================================
# Testing
# ============================================================

if __name__ == "__main__":

    weights = {
        "w1": 1.0,
        "w2": 6.0,
        "wd": 0.2
    }

    # --------------------------------------------------------
    # Test 1: CX-Z-CX cancellation
    # --------------------------------------------------------

    example = {
        "n_qubits": 2,
        "gates": [
            {"gate": "CX", "qubits": [0, 1]},
            {"gate": "Z", "qubits": [0]},
            {"gate": "CX", "qubits": [0,1]},
            {"gate": "Z", "qubits": [1] },
            {"gate": "CX", "qubits": [0,1]},
            {"gate": "Z", "qubits": [1] },
            {"gate": "CX", "qubits": [0,1]}
        ],
    }

    result = optimize(example, weights)

    print("\nTEST 1: CX-Z-CX")

    print("Optimized circuit:")
    print(result)

    print("Evaluation:")
    print(evaluate_optimization(example, result, weights))

    assert equivalent_up_to_global_phase(example, result)

    assert (
        circuit_cost(result, weights)
        <= circuit_cost(example, weights)
    )
    print()
    print("This test demonstrates how our circuit handles CX commutativity, as the Pauli Z gate was commuted with the CX after it to collapse the Identity CX^2 = I")
    # --------------------------------------------------------
    # TEST 2: Combined Level 1 + Level 2
    # --------------------------------------------------------

    example = {
        "n_qubits": 3,
        "gates": [
            {"gate": "H", "qubits": [0]},
            {"gate": "X", "qubits": [1]},
            {"gate": "H", "qubits": [0]},
            {"gate": "CX", "qubits": [1, 2]},
            {"gate": "CX", "qubits": [1, 2]},
            {"gate": "T", "qubits": [0]},
            {"gate": "TDG", "qubits": [0]},
            {"gate": "TDG", "qubits": [0]},
            {"gate": "TDG", "qubits": [0]},
            {"gate": "SDG", "qubits": [0]}
        ]
    }

    result = optimize(example, weights)

    print("\nTEST 2: Combined optimizations")

    print("Optimized circuit:")
    print(result)

    print("Evaluation:")
    print(evaluate_optimization(example, result, weights))

    assert equivalent_up_to_global_phase(example, result)

    assert (
        circuit_cost(result, weights)
        <= circuit_cost(example, weights)
    )

    # --------------------------------------------------------
    # TEST 3: Level 3 triplet identity
    # --------------------------------------------------------

    example = {
        "n_qubits": 2,
        "gates": [
            {"gate": "H", "qubits": [0]},
            {"gate": "X", "qubits": [1]},
            {"gate": "X", "qubits": [0]},
            {"gate": "H", "qubits": [0]}
        ]
    }

    result = optimize(example, weights)

    print("\nTEST 3: HXH -> Z")

    print("Optimized circuit:")
    print(result)

    print("Evaluation:")
    print(evaluate_optimization(example, result, weights))

    assert equivalent_up_to_global_phase(example, result)

    assert (
        circuit_cost(result, weights)
        <= circuit_cost(example, weights)
    )

    # --------------------------------------------------------
    # TEST 4: Randomized equivalence testing
    # --------------------------------------------------------

    print("\nTEST 4: Randomized equivalence testing")

    random.seed(42)

    for test_number in range(100):

        n_qubits = random.randint(1, 4)

        gates = []

        for _ in range(random.randint(1, 20)):

            if n_qubits > 1 and random.random() < 0.3:

                control, target = random.sample(
                    range(n_qubits),
                    2
                )

                gates.append({
                    "gate": "CX",
                    "qubits": [control, target]
                })

            else:

                gate_type = random.choice(
                    tuple(sorted(SINGLE_QUBIT_GATES))
                )

                qubit = random.randrange(n_qubits)

                gates.append({
                    "gate": gate_type,
                    "qubits": [qubit]
                })

        example = {
            "n_qubits": n_qubits,
            "gates": gates
        }

        original_copy = copy.deepcopy(example)

        result = optimize(example, weights)

        assert equivalent_up_to_global_phase(
            example,
            result
        ), f"Equivalence failed on random test {test_number}"

        assert (
            circuit_cost(result, weights)
            <= circuit_cost(example, weights)
        ), f"Cost increased on random test {test_number}"

        assert example == original_copy, (
            f"Original circuit mutated on test {test_number}"
        )

    print("\nAll tests completed successfully.")

# --------------------------------------------------------
# TEST 5: Large quantum gate string simplification
# --------------------------------------------------------
example = {
        "n_qubits": 3,
        "gates": [
            {"gate": "H", "qubits": [1]},
            {"gate": "CX", "qubits": [0,1]},
            {"gate": "X", "qubits": [1]},
            {"gate": "CX", "qubits": [0, 1]},
            {"gate": "H", "qubits": [1]},
            {"gate": "Z", "qubits": [1]},
            {"gate": "CX", "qubits": [1,2]},
            {"gate": "CX", "qubits": [1,2]},
            {"gate": "Z", "qubits": [1]},
            {"gate": "T", "qubits": [2]},
            {"gate": "TDG", "qubits": [2]},
            {"gate": "S", "qubits": [1]},
            {"gate": "S", "qubits": [1]},
        ]
    }

result = optimize(example, weights)

print("\nTEST 5: Large quantum gate string simplification")

print("Optimized circuit:")
print(result)

print("Evaluation:")
print(evaluate_optimization(example, result, weights))

assert equivalent_up_to_global_phase(example, result)

assert (
        circuit_cost(result, weights)
        <= circuit_cost(example, weights)
    )